# Gold: dimension tables for the graph model

Adds two small distinct-value tables so body systems and specialties can be **nodes**
rather than repeated string properties.

| | |
| --- | --- |
| **Reads** | `silver_hpo_terms`, `silver_encounters`, `gold_criteria_definitions` |
| **Writes** | `gold_body_systems`, `gold_specialties` |

## Why these exist

A graph node type needs a key column that **uniquely identifies each node**. Pointing a
`BodySystem` node at `silver_hpo_terms` would offer sixteen rows for seven systems, so the
key would not be unique. These tables are the distinct sets, one row per node.

This is the "embedded entity" pattern from the Fabric graph schema guidance: a column
that represents a shared entity gets extracted into its own node type when you need to
traverse *through* it. Here we do — "which body systems co-occur in surfaced children"
is a traversal, and it is the question a clinician actually asks.

## Nothing else is touched

This notebook only adds tables. It does not rewrite anything the pipeline already
produced, because the evidence contracts and the verified figures were built from those
and a replaced Delta table is a new identity to anything bound to it.

In [ ]:
PIPELINE_RUN_ID = ""

In [ ]:
import notebookutils
from pyspark.sql import functions as F

RUN_ID = PIPELINE_RUN_ID or "local"
_WS = notebookutils.runtime.context["currentWorkspaceId"]
_ONELAKE = notebookutils.conf.get("trident.onelake.endpoint").replace("https://", "")
_LAKEHOUSE_ID = {}


def lake_table(lakehouse, table):
    if lakehouse not in _LAKEHOUSE_ID:
        _LAKEHOUSE_ID[lakehouse] = notebookutils.lakehouse.get(
            lakehouse, workspaceId=_WS).id
    return spark.read.format("delta").load(
        f"abfss://{_WS}@{_ONELAKE}/{_LAKEHOUSE_ID[lakehouse]}/Tables/{table}")


SILVER = "silver_lakehouse"
print("workspace", _WS)

In [ ]:
# ------------------------------------------------------------- body systems
# One row per body system, with how many HPO terms map to it. The count is a property
# a clinician can sanity-check: "craniofacial has three terms" is inspectable.
systems = (lake_table(SILVER, "silver_hpo_terms")
           .groupBy("body_system")
           .agg(F.count("*").cast("int").alias("term_count"),
                F.sort_array(F.collect_list("hpo_label")).alias("terms"))
           .withColumn("terms", F.concat_ws(", ", F.col("terms")))
           .withColumn("run_id", F.lit(RUN_ID))
           .orderBy("body_system"))

systems.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("gold_body_systems")

print(f"gold_body_systems  {systems.count()} rows")
for row in systems.collect():
    print(f"  {row['body_system']:18} {row['term_count']}  {row['terms'][:70]}")

In [ ]:
# --------------------------------------------------------------- specialties
# Distinct conformed specialty names, with the volume behind each. Built from the
# CONFORMED column, not the raw one -- the whole point of silver was that
# "Pediatrics" and "Paediatrics" are one service, and the graph must agree.
specialties = (lake_table(SILVER, "silver_encounters")
               .groupBy("specialty")
               .agg(F.count("*").cast("int").alias("encounter_count"),
                    F.countDistinct("patient_id").cast("int").alias("patient_count"))
               .withColumn("run_id", F.lit(RUN_ID))
               .orderBy(F.desc("encounter_count")))

specialties.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("gold_specialties")

print(f"gold_specialties  {specialties.count()} rows")
for row in specialties.collect():
    print(f"  {row['specialty']:26} encounters={row['encounter_count']:>6,}  "
          f"patients={row['patient_count']:>5,}")

In [ ]:
# ---------------------------------------------------------------- key checks
# A graph node key must be unique, and graph does NOT support schema evolution -- a
# duplicate key found after loading means rebuilding the model from scratch. Cheaper
# to assert it here.
problems = []
for table, key in [("gold_body_systems", "body_system"),
                   ("gold_specialties", "specialty"),
                   ("gold_criteria_definitions", "criterion")]:
    frame = spark.table(table)
    total, distinct = frame.count(), frame.select(key).distinct().count()
    nulls = frame.filter(F.col(key).isNull()).count()
    flag = "ok " if (total == distinct and not nulls) else "FAIL"
    print(f"  {flag} {table}.{key:14} rows={total:>4}  distinct={distinct:>4}  "
          f"nulls={nulls}")
    if total != distinct:
        problems.append(f"{table}.{key} is not unique")
    if nulls:
        problems.append(f"{table}.{key} has {nulls} null(s)")

# Edge source tables must carry both endpoint keys with matching values and types.
edges = [
    ("silver_observations", "patient_id", "hpo_id", SILVER),
    ("silver_hpo_terms", "hpo_id", "body_system", SILVER),
    ("silver_encounters", "patient_id", "encounter_id", SILVER),
    ("silver_encounters", "encounter_id", "specialty", SILVER),
]
print()
for table, source_key, target_key, lakehouse in edges:
    frame = lake_table(lakehouse, table)
    types = dict(frame.dtypes)
    bad = [c for c in (source_key, target_key)
           if c not in frame.columns or types.get(c) != "string"]
    flag = "ok " if not bad else "FAIL"
    print(f"  {flag} edge source {table:22} {source_key} -> {target_key}")
    if bad:
        problems.append(f"{table} missing or mistyped: {bad}")

hits = spark.table("gold_criteria_hits")
print(f"  ok  edge source gold_criteria_hits     patient_id -> criterion "
      f"({hits.count():,} rows)")

if problems:
    for p in problems:
        print("  ", p)
    raise ValueError(f"{len(problems)} graph key problem(s); fix before modelling")
print("\nkeys are unique and every edge source carries both endpoints")